In [ ]:
# ============================================================
# 1. CÀI ĐẶT MÔI TRƯỜNG & THƯ VIỆN
# ============================================================
!pip install pyspark matplotlib seaborn scikit-learn -q

import os
import json
import warnings
import numpy as np
import gc
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import col, when, mean, row_number
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Khởi tạo Spark tối ưu RAM
warnings.filterwarnings("ignore", category=FutureWarning)
spark = SparkSession.builder \
    .appName("EnhancedDepressionModel") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.maxResultSize", "2g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.sparkContext.setCheckpointDir('/tmp/spark-checkpoints')

# ============================================================
# 2. ĐỌC & TIỀN XỬ LÝ DỮ LIỆU
# ============================================================
print("⏳ Khởi tạo Spark Session & Đọc dữ liệu...")
CSV_FILE = "/kaggle/input/datasets/akina484/atn-data-1/Student Depression Dataset.csv"

# Đọc file với phân tách phẩy trước
df = spark.read.csv(CSV_FILE, sep=',', header=True, inferSchema=True)
if len(df.columns) <= 1:
    df = spark.read.csv(CSV_FILE, sep=';', header=True, inferSchema=True)

# Xóa khoảng trắng thừa ở tên cột nếu có
for col_name in df.columns:
    df = df.withColumnRenamed(col_name, col_name.strip())

df = df.withColumn("Gender", when(col("Gender") == "Male", 1).otherwise(0))
df = df.withColumn("Family History of Mental Illness", 
                   when(col("Family History of Mental Illness") == "Yes", 1).otherwise(0))

df = df.withColumn("Sleep Duration",
        when(col("Sleep Duration") == "Less than 5 hours", 0)
       .when(col("Sleep Duration") == "5-6 hours", 1)
       .when(col("Sleep Duration") == "7-8 hours", 2)
       .otherwise(3))

INPUT_COLS = ["Age", "CGPA", "Academic Pressure", "Study Satisfaction", 
              "Financial Stress", "Sleep Duration", "Gender", 
              "Family History of Mental Illness"]
existing_cols = [c for c in INPUT_COLS if c in df.columns]

# Ép kiểu dữ liệu và điền missing values bằng trung vị
for column in existing_cols:
    df = df.withColumn(column, col(column).cast("double"))
    median_val = df.stat.approxQuantile(column, [0.5], 0.001)[0]
    df = df.fillna(median_val, subset=[column])

df = df.withColumn("Depression", col("Depression").cast("integer"))

# ============================================================
# 3. XÂY DỰNG PIPELINE
# ============================================================
assembler = VectorAssembler(inputCols=existing_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=True)

rf = RandomForestClassifier(
    featuresCol="features", 
    labelCol="Depression",
    numTrees=200, 
    maxDepth=15, 
    minInstancesPerNode=2,
    subsamplingRate=0.8,
    seed=42
)

pipeline = Pipeline(stages=[assembler, scaler, rf])

# ============================================================
# 4. K-FOLD CROSS VALIDATION (TÍNH AUC VÀ VẼ ROC)
# ============================================================
K = 5
print(f"\n🚀 Đang chạy K-Fold (K={K}) với Random Forest...")

windowSpec = Window.orderBy("Age")
df = df.withColumn("row_num", row_number().over(windowSpec))
df = df.withColumn("fold", (col("row_num") % K).cast("integer"))
df = df.checkpoint()

fold_metrics = []
n_samples_per_fold = []

# Trình đánh giá AUC của PySpark
evaluator_auc = BinaryClassificationEvaluator(labelCol="Depression", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

plt.figure(figsize=(10, 8)) # Khởi tạo hình vẽ ROC

for i in range(K):
    print(f"--- Đang xử lý Fold {i+1}/{K} ---")
    train_df = df.filter(col("fold") != i).drop("fold", "row_num").checkpoint()
    val_df   = df.filter(col("fold") == i).drop("fold", "row_num")

    # Huấn luyện
    fold_model  = pipeline.fit(train_df)
    
    # Dự đoán
    predictions = fold_model.transform(val_df)

    # Tính toán Accuracy, Precision, Recall, F1
    preds_labels = predictions.select("prediction", col("Depression").cast("double")).rdd.map(list)
    metrics = MulticlassMetrics(preds_labels)
    
    # Tính AUC
    auc_val = evaluator_auc.evaluate(predictions)

    fold_metrics.append([metrics.accuracy, metrics.weightedPrecision, metrics.weightedRecall, metrics.weightedFMeasure(), auc_val])
    n_samples_per_fold.append(val_df.count())
    
    # Chuẩn bị dữ liệu vẽ ROC (Lấy xác suất lớp 1)
    probs_and_labels = predictions.select("probability", "Depression").collect()
    y_prob = [row['probability'][1] for row in probs_and_labels]
    y_test = [row['Depression'] for row in probs_and_labels]
    
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, lw=2, label=f"Fold {i+1} (AUC = {auc_val:.4f})")

    spark.catalog.clearCache()
    gc.collect()

# ============================================================
# 5. IN BẢNG KẾT QUẢ ĐẸP
# ============================================================
avg_stats = np.mean(fold_metrics, axis=0)
std_stats = np.std(fold_metrics,  axis=0)

import pandas as pd
res_df = pd.DataFrame(fold_metrics, columns=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'])
res_df.insert(0, 'Fold', [f"Fold {i+1}" for i in range(K)])
res_df.insert(1, 'Samples', n_samples_per_fold)

avg_row = {
    'Fold': 'AVG', 'Samples': '---',
    'Accuracy': avg_stats[0], 'Precision': avg_stats[1],
    'Recall': avg_stats[2], 'F1-Score': avg_stats[3], 'AUC': avg_stats[4]
}
std_row = {
    'Fold': 'STD', 'Samples': '---',
    'Accuracy': std_stats[0], 'Precision': std_stats[1],
    'Recall': std_stats[2], 'F1-Score': std_stats[3], 'AUC': std_stats[4]
}

res_df = pd.concat([res_df, pd.DataFrame([avg_row, std_row])], ignore_index=True)

print("\n+---------------------------------------------------------------------------------+")
print("| Fold     | Samples   | Accuracy  | Precision | Recall    | F1-Score  | AUC      |")
print("+---------------------------------------------------------------------------------+")
for index, row in res_df.iterrows():
    if row['Fold'] in ['AVG', 'STD']:
        print("+---------------------------------------------------------------------------------+")
    print(f"| {row['Fold']:<8} | {row['Samples']:<9} | {row['Accuracy']:.4f}    | {row['Precision']:.4f}    | {row['Recall']:.4f}    | {row['F1-Score']:.4f}    | {row['AUC']:.4f}   |")
print("+---------------------------------------------------------------------------------+")

# ============================================================
# 6. HOÀN THIỆN BIỂU ĐỒ & LƯU MODEL
# ============================================================
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('ROC Curves for 5-Folds (PySpark Random Forest)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.show()
plt.savefig("/kaggle/working/roc_folds_rf.png", dpi=300, bbox_inches='tight')

print("\n⏳ Đang huấn luyện lại mô hình trên toàn bộ dữ liệu để xuất file...")
final_model = pipeline.fit(df.drop("fold", "row_num"))

save_path = "/kaggle/working/depression_rf_model"
if os.path.exists(save_path):
    import shutil
    shutil.rmtree(save_path)
final_model.save(save_path)

# Lưu Feature Importance
rf_stage = final_model.stages[2]
importances = rf_stage.featureImportances.toArray().tolist()

model_info = {
    "model_type": "Random Forest (Enhanced)",
    "avg_accuracy": float(avg_stats[0]),
    "features": existing_cols,
    "importances": importances
}

with open("/kaggle/working/model_summary.json", "w") as f:
    json.dump(model_info, f, indent=4)

print(f"💾 Đã lưu model tại: {save_path}")
print("👉 Bạn hãy check cột Output bên phải Kaggle để tải thư mục model về nhé!")
spark.stop()